In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import Required Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, LeakyReLU, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd
from tabulate import tabulate

Define Dataset Paths

In [ ]:
train_dir = "/content/drive/MyDrive/HA_EXP/train"
test_dir = "/content/drive/MyDrive/HA_EXP/test"

Define Image Parameters

In [ ]:
img_size = (128,128)
batch_size = 32
epochs = 10

Image Preprocessing using ImageDataGenerator

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

Load Training and Testing Dataset

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)

Found 557 images belonging to 2 classes.
Found 140 images belonging to 2 classes.


Define Activation Functions and Optimizers for Comparison

In [ ]:
activations = ["relu","sigmoid","leakyrelu"]
optimizers = ["Adam","Adagrad","Adamax"]

Initialize Result Storage

In [ ]:
results = []

Train CNN Model with Different Activation Functions and Optimizers

In [ ]:
for act in activations:
    for opt_name in optimizers:

        if opt_name == "Adam":
            opt = tf.keras.optimizers.Adam()
        elif opt_name == "Adagrad":
            opt = tf.keras.optimizers.Adagrad()
        else:
            opt = tf.keras.optimizers.Adamax()

        model = Sequential()
        model.add(Input(shape=(128,128,3)))
        model.add(Conv2D(32,(3,3)))

        if act == "leakyrelu":
            model.add(LeakyReLU())
        else:
            model.add(tf.keras.layers.Activation(act))

        model.add(MaxPooling2D((2,2)))
        model.add(Conv2D(64,(3,3)))

        if act == "leakyrelu":
            model.add(LeakyReLU())
        else:
            model.add(tf.keras.layers.Activation(act))

        model.add(MaxPooling2D((2,2)))
        model.add(Flatten())
        model.add(Dense(128))

        if act == "leakyrelu":
            model.add(LeakyReLU())
        else:
            model.add(tf.keras.layers.Activation(act))

        model.add(Dense(1,activation='sigmoid'))

        model.compile(
            optimizer=opt,
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        history = model.fit(
            train_generator,
            epochs=epochs,
            validation_data=test_generator,
            verbose=0
        )

        train_acc = history.history['accuracy'][-1]
        val_acc = history.history['val_accuracy'][-1]

        results.append([act,opt_name,round(train_acc,3),round(val_acc,3)])

Convert Results to DataFrame

In [ ]:
df = pd.DataFrame(
    results,
    columns=["Activation","Optimizer","Train Accuracy","Validation Accuracy"]
)

Display Comparison Table

In [ ]:
print("\nMODEL COMPARISON TABLE\n")
print(tabulate(df,headers='keys',tablefmt='grid',showindex=False))


MODEL COMPARISON TABLE

+--------------+-------------+------------------+-----------------------+
| Activation   | Optimizer   |   Train Accuracy |   Validation Accuracy |
+==============+=============+==================+=======================+
| relu         | Adam        |            1     |                 0.643 |
+--------------+-------------+------------------+-----------------------+
| relu         | Adagrad     |            0.589 |                 0.557 |
+--------------+-------------+------------------+-----------------------+
| relu         | Adamax      |            0.934 |                 0.664 |
+--------------+-------------+------------------+-----------------------+
| sigmoid      | Adam        |            0.488 |                 0.5   |
+--------------+-------------+------------------+-----------------------+
| sigmoid      | Adagrad     |            0.447 |                 0.5   |
+--------------+-------------+------------------+-----------------------+
| sigmoid    